In [2]:
import pandas as pd

x_train = pd.read_csv('x_train_final.csv')
y_train = pd.read_csv('y_train_final.csv')
x_test  = pd.read_csv('x_test_final.csv')

print(f"\nx_train: {x_train.shape}")
print(f"y_train: {y_train.shape}")
print(f"x_test: {x_test.shape}")


x_train: (667264, 12)
y_train: (667264, 2)
x_test: (20657, 11)


In [3]:
# EDA
print("=== X_TRAIN ===")
print(f"Shape: {x_train.shape}")
print(f"Colonnes: {list(x_train.columns)}")
print(f"\nTypes:\n{x_train.dtypes}")
print(f"\nValeurs uniques:")
for col in x_train.columns:
    print(f"  {col}: {x_train[col].nunique()}")
print(f"\n{x_train.head()}")
print(f"\n{x_train.describe()}")

print("\n=== Y_TRAIN ===")
print(f"Shape: {y_train.shape}")
print(f"\n{y_train.describe()}")

print("\n=== X_TEST ===")
print(f"Shape: {x_test.shape}")
print(f"\n{x_test.head()}")

=== X_TRAIN ===
Shape: (667264, 12)
Colonnes: ['Unnamed: 0.1', 'Unnamed: 0', 'train', 'gare', 'date', 'arret', 'p2q0', 'p3q0', 'p4q0', 'p0q2', 'p0q3', 'p0q4']

Types:
Unnamed: 0.1      int64
Unnamed: 0        int64
train               str
gare                str
date                str
arret             int64
p2q0            float64
p3q0            float64
p4q0            float64
p0q2            float64
p0q3            float64
p0q4            float64
dtype: object

Valeurs uniques:
  Unnamed: 0.1: 667264
  Unnamed: 0: 667264
  train: 37544
  gare: 84
  date: 91
  arret: 36
  p2q0: 120
  p3q0: 123
  p4q0: 121
  p0q2: 121
  p0q3: 123
  p0q4: 123

   Unnamed: 0.1  Unnamed: 0   train gare        date  arret  p2q0  p3q0  p4q0  \
0             0           0  VBXNMF  KYF  2023-04-03      8   0.0   0.0   1.0   
1             1           1  VBXNMF  JLR  2023-04-03      9   0.0   0.0   0.0   
2             2           2  VBXNMF  EOH  2023-04-03     10  -1.0   0.0   0.0   
3             3        

In [4]:
# Enlever les outliers avec IQR (moins agressif: 3x au lieu de 1.5x)
y_vals = y_train["p0q0"]

Q1 = y_vals.quantile(0.25)
Q3 = y_vals.quantile(0.75)
IQR = Q3 - Q1

# Limites plus larges (3x IQR) pour garder plus de données
lower_bound = Q1 - 3 * IQR
upper_bound = Q3 + 3 * IQR

print(f"Q1: {Q1}, Q3: {Q3}, IQR: {IQR}")
print(f"Lower bound: {lower_bound}, Upper bound: {upper_bound}")

# Garder les indices valides
valid_idx = (y_vals >= lower_bound) & (y_vals <= upper_bound)

# Filtrer les données
x_train = x_train[valid_idx].reset_index(drop=True)
y_train = y_train[valid_idx].reset_index(drop=True)

print(f"\nAprès suppression outliers:")
print(f"x_train: {x_train.shape}")
print(f"y_train: {y_train.shape}")
print(f"Nombre d'outliers supprimés: {(~valid_idx).sum()}")

Q1: -1.0, Q3: 1.0, IQR: 2.0
Lower bound: -7.0, Upper bound: 7.0

Après suppression outliers:
x_train: (664182, 12)
y_train: (664182, 2)
Nombre d'outliers supprimés: 3082


In [5]:
import pandas as pd
import numpy as np
import networkx as nx
from sklearn.preprocessing import LabelEncoder

y = y_train["p0q0"]

full = pd.concat([x_train, x_test], axis=0)
full = full.drop(columns=["Unnamed: 0", "Unnamed: 0.1"], errors="ignore")

# date features
full["date"] = pd.to_datetime(full["date"])
full["jour"] = full["date"].dt.day
full["mois"] = full["date"].dt.month
full["jour_semaine"] = full["date"].dt.dayofweek  # gardé pour le target encoding

# ===== GRAPHE DIRIGÉ DU RÉSEAU (networkx) =====
G = nx.DiGraph()
for _, grp in full.groupby(["train", "date"]):
    gares = grp.sort_values("arret")["gare"].values
    for a, b in zip(gares[:-1], gares[1:]):
        G.add_edge(a, b)

# Features extraites du graphe dirigé
in_degree = dict(G.in_degree())
out_degree = dict(G.out_degree())
betweenness = nx.betweenness_centrality(G)

full["gare_in_degree"] = full["gare"].map(in_degree).fillna(0)
full["gare_out_degree"] = full["gare"].map(out_degree).fillna(0)
full["gare_betweenness"] = full["gare"].map(betweenness).fillna(0)

print(f"DiGraph: {G.number_of_nodes()} gares, {G.number_of_edges()} arcs")

# ===== FEATURE ENGINEERING =====
cols_gare = ["p2q0", "p3q0", "p4q0"]
cols_train = ["p0q2", "p0q3", "p0q4"]
cols_retard = cols_gare + cols_train

full["mean_retard"] = full[cols_retard].mean(axis=1)
full["std_retard"] = full[cols_retard].std(axis=1)
full["max_retard"] = full[cols_retard].max(axis=1)

full["mean_retard_gare_hist"] = full[cols_gare].mean(axis=1)
full["mean_retard_train_hist"] = full[cols_train].mean(axis=1)

full["trend_gare"] = full["p2q0"] - full["p4q0"]
full["trend_gare_2"] = full["p2q0"] - full["p3q0"]

full["sum_retard_gare"] = full[cols_gare].sum(axis=1)
full["sum_retard_train"] = full[cols_train].sum(axis=1)

# NOUVELLES FEATURES
full["train_median"] = full[cols_train].median(axis=1)
full["mean_retard_global"] = y.mean()

full["diff_gare_train"] = full["mean_retard_gare_hist"] - full["mean_retard_train_hist"]

full["retard_x_arret"] = full["mean_retard"] * full["arret"]
full["arret_squared"] = full["arret"] ** 2

# nb_arrets_train (basé sur arret, pas sur la target → pas de leakage)
train_part = full.iloc[:len(y)]
nb_arrets = train_part.groupby("train")["arret"].max().rename("nb_arrets_train")
full = full.merge(nb_arrets, on="train", how="left")
full["position_norm"] = full["arret"] / full["nb_arrets_train"].clip(lower=1)

# freq_trains_par_jour (fréquence moyenne des trains par jour pour chaque train)
train_date_counts = train_part.groupby(["train", "date"]).size().reset_index(name="freq")
freq_mean = train_date_counts.groupby("train")["freq"].mean().rename("freq_trains_par_jour")
full = full.merge(freq_mean, on="train", how="left")
full["freq_trains_par_jour"] = full["freq_trains_par_jour"].fillna(0)

_date = full["date"]
full = full.drop(columns=["date"]).fillna(0)
full["date"] = _date  # gardé pour rolling 7j (sera droppé dans add_target_features)

# Label encoding
le_train = LabelEncoder()
full["train"] = le_train.fit_transform(full["train"])
le_gare = LabelEncoder()
full["gare"] = le_gare.fit_transform(full["gare"])

# re-split (SANS les features target-encoded, elles seront ajoutées après le split)
x_train = full.iloc[:len(y)]
x_test = full.iloc[len(y):]

print(f"x_train: {x_train.shape}, x_test: {x_test.shape}")
print("Colonnes:", list(x_train.columns))

DiGraph: 84 gares, 1081 arcs
x_train: (664182, 33), x_test: (20657, 33)
Colonnes: ['train', 'gare', 'arret', 'p2q0', 'p3q0', 'p4q0', 'p0q2', 'p0q3', 'p0q4', 'jour', 'mois', 'jour_semaine', 'gare_in_degree', 'gare_out_degree', 'gare_betweenness', 'mean_retard', 'std_retard', 'max_retard', 'mean_retard_gare_hist', 'mean_retard_train_hist', 'trend_gare', 'trend_gare_2', 'sum_retard_gare', 'sum_retard_train', 'train_median', 'mean_retard_global', 'diff_gare_train', 'retard_x_arret', 'arret_squared', 'nb_arrets_train', 'position_norm', 'freq_trains_par_jour', 'date']


In [6]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error

def smoothed_mean(group_means, group_counts, global_mean, alpha=20):
    """Smoothing: quand peu d'observations, on se rapproche de la moyenne globale.
    smoothed = (n * mean_locale + alpha * mean_globale) / (n + alpha)"""
    return (group_counts * group_means + alpha * global_mean) / (group_counts + alpha)

def compute_stats_7j(data, group_col):
    """Rolling 7j sur la target — même logique que le pote.
    Pour chaque date, on regarde les 7 jours PRÉCÉDENTS (strict) pour éviter le leakage."""
    records = []
    data = data.copy()
    data["date"] = pd.to_datetime(data["date"])
    for date in sorted(data["date"].unique()):
        date_min = date - pd.Timedelta(days=7)
        fenetre = data[(data["date"] > date_min) & (data["date"] < date)]
        if len(fenetre) == 0:
            continue
        stats = fenetre.groupby(group_col)["_y"].agg(
            **{f"{group_col}_mean_7j": "mean",
               f"{group_col}_std_7j":  "std"}
        ).reset_index()
        stats["date"] = date
        records.append(stats)
    if records:
        return pd.concat(records, ignore_index=True)
    return pd.DataFrame()

def add_target_features(df, x_fit, y_fit, alpha=20):
    """Ajoute les features target-encoded avec smoothing + rolling 7j.
    Calculées sur (x_fit, y_fit), appliquées à df."""
    result = df.reset_index(drop=True).copy()
    fit = x_fit[["gare", "train", "arret", "jour_semaine", "date"]].copy().reset_index(drop=True)
    fit["_y"] = y_fit.values
    global_mean = fit["_y"].mean()

    # --- Smoothed target encoding ---
    # Par gare
    g = fit.groupby("gare")["_y"]
    gare_mean = g.mean()
    gare_count = g.count()
    gare_smoothed = smoothed_mean(gare_mean, gare_count, global_mean, alpha)
    result["mean_retard_gare"] = result["gare"].map(gare_smoothed).fillna(global_mean)

    gare_std = g.std().fillna(0)
    result["std_retard_gare"] = result["gare"].map(gare_std).fillna(0)

    # Par train
    g = fit.groupby("train")["_y"]
    train_mean = g.mean()
    train_count = g.count()
    train_smoothed = smoothed_mean(train_mean, train_count, global_mean, alpha)
    result["mean_retard_train"] = result["train"].map(train_smoothed).fillna(global_mean)

    # Par arret
    g = fit.groupby("arret")["_y"]
    arret_mean = g.mean()
    arret_count = g.count()
    arret_smoothed = smoothed_mean(arret_mean, arret_count, global_mean, alpha)
    result["mean_retard_arret"] = result["arret"].map(arret_smoothed).fillna(global_mean)

    # Par gare + jour_semaine (smoothing aussi)
    gj = fit.groupby(["gare", "jour_semaine"])["_y"].agg(["mean", "count"]).reset_index()
    gj["mean_retard_gare_jour"] = smoothed_mean(gj["mean"], gj["count"], global_mean, alpha)
    gj = gj[["gare", "jour_semaine", "mean_retard_gare_jour"]]
    result = result.merge(gj, on=["gare", "jour_semaine"], how="left")
    result["mean_retard_gare_jour"] = result["mean_retard_gare_jour"].fillna(global_mean)

    # --- Rolling 7 jours sur la target (comme le pote) ---
    result["date"] = pd.to_datetime(result["date"])

    stats_train_7j = compute_stats_7j(fit, "train")
    stats_gare_7j  = compute_stats_7j(fit, "gare")

    if not stats_train_7j.empty:
        result = result.merge(stats_train_7j, on=["train", "date"], how="left")
    else:
        result["train_mean_7j"] = np.nan
        result["train_std_7j"] = np.nan

    if not stats_gare_7j.empty:
        result = result.merge(stats_gare_7j, on=["gare", "date"], how="left")
    else:
        result["gare_mean_7j"] = np.nan
        result["gare_std_7j"] = np.nan

    # Fallback NaN → smoothed target mean (comme le pote)
    result["train_mean_7j"] = result["train_mean_7j"].fillna(result["mean_retard_train"])
    result["train_std_7j"]  = result["train_std_7j"].fillna(0)
    result["gare_mean_7j"]  = result["gare_mean_7j"].fillna(result["mean_retard_gare"])
    result["gare_std_7j"]   = result["gare_std_7j"].fillna(0)

    # Supprimer jour_semaine et date
    result = result.drop(columns=["jour_semaine", "date"])
    return result

# 1. Split
x_tr_raw, x_val_raw, y_tr, y_val = train_test_split(x_train, y, test_size=0.2, random_state=42)

# 2. Target encoding + rolling 7j (calculé UNIQUEMENT sur le train fold → pas de leakage)
print("Computing target features (train fold)...")
x_tr = add_target_features(x_tr_raw, x_tr_raw, y_tr, alpha=20)
print("Computing target features (val fold)...")
x_val = add_target_features(x_val_raw, x_tr_raw, y_tr, alpha=20)

# 3. Modèle
model = RandomForestRegressor(
    n_estimators=500,
    max_depth=20,
    min_samples_split=8,
    min_samples_leaf=3,
    max_features='sqrt',
    n_jobs=-1,
    random_state=42
)
model.fit(x_tr, y_tr)

# 4. Évaluation
pred_tr = model.predict(x_tr)
pred_val = model.predict(x_val)

mae_tr = mean_absolute_error(y_tr, pred_tr)
mae_val = mean_absolute_error(y_val, pred_val)

print(f"MAE train:      {mae_tr:.4f}")
print(f"MAE validation: {mae_val:.4f}")
print(f"Écart train/val: {abs(mae_tr - mae_val):.4f}")
print(f"Features ({x_tr.shape[1]}): {list(x_tr.columns)}")

if mae_tr < mae_val * 0.7:
    print("⚠️ Possible overfitting")
else:
    print("✅ Pas d'overfitting significatif")

Computing target features (train fold)...
Computing target features (val fold)...
MAE train:      0.4831
MAE validation: 0.6304
Écart train/val: 0.1473
Features (40): ['train', 'gare', 'arret', 'p2q0', 'p3q0', 'p4q0', 'p0q2', 'p0q3', 'p0q4', 'jour', 'mois', 'gare_in_degree', 'gare_out_degree', 'gare_betweenness', 'mean_retard', 'std_retard', 'max_retard', 'mean_retard_gare_hist', 'mean_retard_train_hist', 'trend_gare', 'trend_gare_2', 'sum_retard_gare', 'sum_retard_train', 'train_median', 'mean_retard_global', 'diff_gare_train', 'retard_x_arret', 'arret_squared', 'nb_arrets_train', 'position_norm', 'freq_trains_par_jour', 'mean_retard_gare', 'std_retard_gare', 'mean_retard_train', 'mean_retard_arret', 'mean_retard_gare_jour', 'train_mean_7j', 'train_std_7j', 'gare_mean_7j', 'gare_std_7j']
✅ Pas d'overfitting significatif


In [7]:
from sklearn.model_selection import KFold

# Cross-validation avec target encoding + rolling 7j (pas de leakage) — 5 folds
kf = KFold(n_splits=5, shuffle=True, random_state=42)
cv_maes = []

for fold, (tr_idx, val_idx) in enumerate(kf.split(x_train)):
    x_cv_tr, x_cv_val = x_train.iloc[tr_idx], x_train.iloc[val_idx]
    y_cv_tr, y_cv_val = y.iloc[tr_idx], y.iloc[val_idx]

    print(f"Fold {fold+1}: computing features...")
    x_cv_tr_enc = add_target_features(x_cv_tr, x_cv_tr, y_cv_tr, alpha=20)
    x_cv_val_enc = add_target_features(x_cv_val, x_cv_tr, y_cv_tr, alpha=20)

    model_cv = RandomForestRegressor(
        n_estimators=500, max_depth=20, min_samples_split=8,
        min_samples_leaf=3, max_features='sqrt', n_jobs=-1, random_state=42
    )
    model_cv.fit(x_cv_tr_enc, y_cv_tr)

    mae = mean_absolute_error(y_cv_val, model_cv.predict(x_cv_val_enc))
    cv_maes.append(mae)
    print(f"Fold {fold+1}: MAE = {mae:.4f}")

print(f"\nMAE moyen: {np.mean(cv_maes):.4f} ± {np.std(cv_maes):.4f}")

Fold 1: computing features...
Fold 1: MAE = 0.6306
Fold 2: computing features...
Fold 2: MAE = 0.6344
Fold 3: computing features...
Fold 3: MAE = 0.6320
Fold 4: computing features...
Fold 4: MAE = 0.6320
Fold 5: computing features...
Fold 5: MAE = 0.6368

MAE moyen: 0.6331 ± 0.0022


In [8]:
# Feature Importance (utilise x_tr qui a les bonnes colonnes)
feature_importance = pd.DataFrame({
    'feature': x_tr.columns,
    'importance': model.feature_importances_
}).sort_values('importance', ascending=False)

print("\n--- Feature Importance ---")
print(feature_importance)

import matplotlib.pyplot as plt
plt.figure(figsize=(10, 6))
plt.barh(feature_importance['feature'], feature_importance['importance'])
plt.xlabel('Importance')
plt.title('Feature Importance - RandomForest')
plt.gca().invert_yaxis()
plt.tight_layout()
plt.savefig('feature_importance.png', dpi=100, bbox_inches='tight')
plt.show()


--- Feature Importance ---
                   feature  importance
38            gare_mean_7j    0.121533
35   mean_retard_gare_jour    0.087348
31        mean_retard_gare    0.081273
33       mean_retard_train    0.056418
36           train_mean_7j    0.054153
18  mean_retard_train_hist    0.041404
22        sum_retard_train    0.041332
25         diff_gare_train    0.032529
29           position_norm    0.031240
23            train_median    0.028137
39             gare_std_7j    0.025223
3                     p2q0    0.022001
27           arret_squared    0.021492
2                    arret    0.021395
20            trend_gare_2    0.019569
34       mean_retard_arret    0.019557
26          retard_x_arret    0.019253
6                     p0q2    0.019072
15              std_retard    0.017349
32         std_retard_gare    0.016837
0                    train    0.015353
28         nb_arrets_train    0.015261
30    freq_trains_par_jour    0.014887
12         gare_out_degree    0.0147

ModuleNotFoundError: No module named 'matplotlib'

In [ ]:
# Filtrer les features faibles (importance < 0.005) + train_std_7j inutile
features_to_keep = feature_importance[feature_importance['importance'] >= 0.005]['feature'].tolist()
print(f"Garder {len(features_to_keep)} features / {len(x_tr.columns)}")
print(f"Features sélectionnées: {features_to_keep}")

# Re-split avec les features filtrées et re-train pour évaluation
x_tr_filtered = x_tr[features_to_keep]
x_val_filtered = x_val[features_to_keep]

model_filtered = RandomForestRegressor(
    n_estimators=500,
    max_depth=20,
    min_samples_split=8,
    min_samples_leaf=3,
    max_features='sqrt',
    n_jobs=-1,
    random_state=42
)
model_filtered.fit(x_tr_filtered, y_tr)

pred_tr_filtered = model_filtered.predict(x_tr_filtered)
pred_val_filtered = model_filtered.predict(x_val_filtered)

mae_tr_filtered = mean_absolute_error(y_tr, pred_tr_filtered)
mae_val_filtered = mean_absolute_error(y_val, pred_val_filtered)

print(f"\n--- Après filtrage des features ---")
print(f"MAE train:      {mae_tr_filtered:.4f} (avant: {mae_tr:.4f})")
print(f"MAE validation: {mae_val_filtered:.4f} (avant: {mae_val:.4f})")
print(f"Amélioration: {mae_val - mae_val_filtered:.4f}")

Garder 38 features / 40
Features sélectionnées: ['gare_mean_7j', 'mean_retard_gare_jour', 'mean_retard_gare', 'mean_retard_train', 'train_mean_7j', 'mean_retard_train_hist', 'sum_retard_train', 'diff_gare_train', 'position_norm', 'train_median', 'gare_std_7j', 'p2q0', 'arret_squared', 'arret', 'trend_gare_2', 'mean_retard_arret', 'retard_x_arret', 'p0q2', 'std_retard', 'std_retard_gare', 'train', 'nb_arrets_train', 'freq_trains_par_jour', 'gare_out_degree', 'gare', 'trend_gare', 'gare_betweenness', 'p0q4', 'gare_in_degree', 'mean_retard', 'mean_retard_gare_hist', 'sum_retard_gare', 'jour', 'p0q3', 'p3q0', 'p4q0', 'max_retard', 'mois']

--- Après filtrage des features ---
MAE train:      0.4814 (avant: 0.4831)
MAE validation: 0.6305 (avant: 0.6304)
Amélioration: -0.0000


In [ ]:
# Target encoding + rolling 7j sur TOUT le train, appliqué au train et au test
print("Computing final features...")
x_train_final = add_target_features(x_train, x_train, y, alpha=20)
x_test_final = add_target_features(x_test, x_train, y, alpha=20)

# Filtrer avec les features sélectionnées
x_train_final_filtered = x_train_final[features_to_keep]
x_test_final_filtered = x_test_final[features_to_keep]

model_final = RandomForestRegressor(
    n_estimators=500,
    max_depth=20,
    min_samples_split=8,
    min_samples_leaf=3,
    max_features='sqrt',
    n_jobs=-1,
    random_state=42
)
model_final.fit(x_train_final_filtered, y)

pred_test = model_final.predict(x_test_final_filtered)

# Arrondi à l'entier (comme le pote)
pred_test_rounded = np.round(pred_test).astype(int)

submission = pd.DataFrame({"p0q0": pred_test_rounded})
submission.to_csv("submission_v16_filtered.csv", index=True)

print(f"submission_v16_filtered.csv créé avec {len(submission)} lignes")
print(submission.head())
print(f"\nStats prédictions:\n{submission.describe()}")

Computing final features...
submission_v16_filtered.csv créé avec 20657 lignes
   p0q0
0     0
1     0
2     0
3     0
4     0

Stats prédictions:
               p0q0
count  20657.000000
mean      -0.070533
std        0.684007
min       -3.000000
25%        0.000000
50%        0.000000
75%        0.000000
max        2.000000


In [ ]:
# Configuration GPU - Utiliser le GPU 5070Ti
import tensorflow as tf
import os

# Lister les GPUs disponibles
gpus = tf.config.list_physical_devices('GPU')
print(f"GPUs détectés: {len(gpus)}")
for i, gpu in enumerate(gpus):
    print(f"  GPU {i}: {gpu}")

# Usar le premier GPU disponible (et unique) = 5070Ti
GPU_INDEX = 0
if len(gpus) > 0:
    tf.config.set_visible_devices(gpus[GPU_INDEX], 'GPU')
    print(f"\n✓ GPU {GPU_INDEX} utilisé: {gpus[GPU_INDEX]}")
else:
    print("⚠ Aucun GPU disponible!")

GPUs détectés: 1
  GPU 0: PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')

✓ GPU 0 utilisé: PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')


In [ ]:
# Modèle multimodal simple
from tensorflow.keras import Model
from tensorflow.keras.layers import Dense, Input, Concatenate

class MyModel(tf.keras.Model):
    def __init__(self):
        super().__init__()
        self.dense1 = Dense(4, activation="relu")
        self.dense2 = Dense(4, activation="relu")
        self.dense3 = Dense(1, activation="linear")
    
    def call(self, inputs):
        text, image = inputs
        x1 = self.dense1(text)
        x2 = self.dense2(image)
        x = tf.concat([x1, x2], axis=-1)
        return self.dense3(x)

# Instantier et compiler le modèle
with tf.device(f'/GPU:{GPU_INDEX}'):
    model = MyModel()
    model.compile(optimizer='adam', loss='mse', metrics=['mae'])
    print("✓ Modèle créé et compilé sur GPU 5070Ti")

✓ Modèle créé et compilé sur GPU 5070Ti


In [ ]:
# Entraînement du modèle multimodal sur GPU 5070Ti
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split

# Préparer les données
print("Préparation des données...")
X = x_train_final_filtered.values
y_model = y.values.reshape(-1, 1)

# Splitter les features en deux modalités (moitié-moitié)
n_features = X.shape[1]
split_idx = n_features // 2

X_text = X[:, :split_idx]  # Première moitié
X_image = X[:, split_idx:]  # Deuxième moitié

# Normaliser
scaler_text = StandardScaler()
scaler_image = StandardScaler()
X_text = scaler_text.fit_transform(X_text)
X_image = scaler_image.fit_transform(X_image)

# Splitter train/val
X_text_train, X_text_val, X_image_train, X_image_val, y_train_model, y_val_model = train_test_split(
    X_text, X_image, y_model, test_size=0.2, random_state=42
)

print(f"X_text_train: {X_text_train.shape}, X_image_train: {X_image_train.shape}")
print(f"y_train: {y_train_model.shape}")

# Entraîner sur GPU
print("\n🚀 Entraînement sur GPU 5070Ti...")
with tf.device(f'/GPU:{GPU_INDEX}'):
    history = model.fit(
        [X_text_train, X_image_train], y_train_model,
        validation_data=([X_text_val, X_image_val], y_val_model),
        epochs=20,
        batch_size=32,
        verbose=1
    )

# Évaluation
val_loss, val_mae = model.evaluate([X_text_val, X_image_val], y_val_model, verbose=0)
print(f"\n✅ Entraînement terminé!")
print(f"Validation Loss: {val_loss:.4f}")
print(f"Validation MAE: {val_mae:.4f}")

Préparation des données...


NameError: name 'x_train_final_filtered' is not defined